# Quick Start

This page provides code examples for basic hyperalignment operations, starting with a two-subject, single-ROI Procrustes alignment. It then covers common-template construction, transformation matrix estimation, and applying the resulting alignment.

## 1. Two-Subject Example: Pairwise Procrustes Rotation

This example provides a minimal two-subject demonstration of Procrustes-based alignment. `sub1` is used as the source data matrix, `sub2` is used as the target data matrix, and `procrustes()` returns the T matrix that maps `sub1` into the target space. The aligned data are obtained with matrix multiplication: `sub1_aligned = sub1 @ T_sub1_to_sub2`.

In [ ]:
from pathlib import Path
import numpy as np

from fmriha.src.procrustes import procrustes
from fmriha.src.local_template import compute_template


# ---------------------------------------------------------------------
# Set paths and load two subjects from one ROI or local voxel set
# ---------------------------------------------------------------------
subject_dir = Path("/path/to/raw_data/subject_sl")

# Each array has shape: n_timepoints x n_vertices
sub1 = np.load(subject_dir / "sub1_sl.npy")
sub2 = np.load(subject_dir / "sub2_sl.npy")

print(f"Subject 1 data shape: {sub1.shape}")
print(f"Subject 2 data shape: {sub2.shape}")


# ---------------------------------------------------------------------
# Estimate and apply the Procrustes T matrix
# ---------------------------------------------------------------------
T_sub1_to_sub2 = procrustes(
    X=sub1,
    Y=sub2,
    reflection=True,
    scaling=False,
)

sub1_aligned_to_sub2 = sub1 @ T_sub1_to_sub2

print(f"T matrix shape: {T_sub1_to_sub2.shape}")
print(f"Aligned subject 1 shape: {sub1_aligned_to_sub2.shape}")

Subject 1 data shape: (470, 120)
Subject 2 data shape: (470, 120)
T matrix shape: (120, 120)
Aligned subject 1 shape: (470, 120)


## 2. Minimal Hyperalignment Workflow

This example introduces a minimal implementation of the full hyperalignment workflow. The workflow consists of three core steps:

● **Template estimation**: estimate a shared template from training data.  
● **T-matrix estimation**: estimate subject-specific T matrices that map each subject into the template space.  
● **Alignment application**: apply the estimated T matrices to obtain aligned data.

In this example, the toolbox demonstrates two core alignment algorithms: **Procrustes rotation** and **PCA**. These algorithms provide different ways to estimate the common template and the corresponding subject-specific transformations. After a T matrix has been estimated, aligned data can be obtained with matrix multiplication, such as `subject_data @ subject_T`.

In a real hyperalignment workflow, we strongly recommend separating the data used for template estimation, T-matrix estimation, and final alignment whenever possible, do <span style="color: red; font-weight: bold;">NOT</span> using the same data from the same subjects for all three steps. Common strategies include:

1. **Response-data split**: split the response data into two segments, estimate the template and T matrices using the first segment, and apply the T matrices to the second segment.

2. **Subject split**: use one group of subjects to estimate the template, then estimate T matrices for a separate group of subjects using the same type of data, and apply these T matrices to other data from the test subjects.

3. **Modality split**: when the number of subjects or scan duration is limited, use a connectivity hyperalignment framework to estimate a connectivity-based template and T matrices, and then apply the resulting transformations to response data.

### Procrustes-Based Common Template

First, load the subjects reserved for template construction. Stack these matrices into `template_dss` with shape `n_subjects x n_timepoints x n_vertices`, then call `compute_template(..., kind="procrustes")`. These template subjects are not reused below for estimating T matrices or producing aligned data.

In [ ]:
# ---------------------------------------------------------------------
# Load subjects reserved for common-template construction
# ---------------------------------------------------------------------
sub3 = np.load(subject_dir / "sub3_sl.npy")

print(f"Subject 3 data shape: {sub3.shape}")

# Shape: n_subjects x n_timepoints x n_vertices
template_dss = np.stack([sub1, sub2, sub3], axis=0)
print(f"Template data stack shape: {template_dss.shape}")


# ---------------------------------------------------------------------
# Construct a Procrustes-based common template
# ---------------------------------------------------------------------
M_pr = compute_template(
    dss=template_dss,
    sl=None,
    kind="procrustes",
)

print(f"Procrustes template shape: {M_pr.shape}")

Subject 3 data shape: (470, 120)
Template data stack shape: (3, 470, 120)
Procrustes template shape: (470, 120)


### PCA-Based Common Template

For a PCA-based common template, use `kind="pca"`. Setting `common_topography=True` returns the template in the original voxel or vertex space.

In [ ]:
# ---------------------------------------------------------------------
# Construct a PCA-based common template
# ---------------------------------------------------------------------
M_pca = compute_template(
    dss=template_dss,
    sl=None,
    kind="pca",
    common_topography=True,
)

print(f"PCA template shape: {M_pca.shape}")

PCA template shape: (470, 120)


### Transformation Matrix (T Matrix)

Estimate one T matrix per subject by aligning that subject's raw data to the selected common template. Here, the T matrices are estimated for a second group of subjects, separate from the subjects used to construct the template.

In [ ]:
# ---------------------------------------------------------------------
# Load subjects reserved for T-matrix estimation and alignment
# ---------------------------------------------------------------------
sub4 = np.load(subject_dir / "sub4_sl.npy")
sub5 = np.load(subject_dir / "sub5_sl.npy")
sub6 = np.load(subject_dir / "sub6_sl.npy")

print(f"Subject 4 data shape: {sub4.shape}")
print(f"Subject 5 data shape: {sub5.shape}")
print(f"Subject 6 data shape: {sub6.shape}")

# ---------------------------------------------------------------------
# Estimate subject-specific T matrices for the Procrustes-based common space
# ---------------------------------------------------------------------
sub4_pr_T = procrustes(X=sub4, Y=M_pr, reflection=True, scaling=False)
sub5_pr_T = procrustes(X=sub5, Y=M_pr, reflection=True, scaling=False)
sub6_pr_T = procrustes(X=sub6, Y=M_pr, reflection=True, scaling=False)

print(f"Subject 4 Procrustes T shape: {sub4_pr_T.shape}")


# ---------------------------------------------------------------------
# Estimate subject-specific T matrices for the PCA-based common space
# ---------------------------------------------------------------------
sub4_pca_T = procrustes(X=sub4, Y=M_pca, reflection=True, scaling=False)
sub5_pca_T = procrustes(X=sub5, Y=M_pca, reflection=True, scaling=False)
sub6_pca_T = procrustes(X=sub6, Y=M_pca, reflection=True, scaling=False)

print(f"Subject 4 PCA T shape: {sub4_pca_T.shape}")

Subject 4 data shape: (470, 120)
Subject 5 data shape: (470, 120)
Subject 6 data shape: (470, 120)
Subject 4 Procrustes T shape: (120, 120)
Subject 4 PCA T shape: (120, 120)


### Alignment

Apply each subject-specific T matrix with standard matrix multiplication. The resulting matrices are expressed in the selected common space. This step uses the same second group of subjects used for T-matrix estimation, while the template still comes from the separate template group above.

In [ ]:
# ---------------------------------------------------------------------
# Align each subject to the Procrustes-based common space
# ---------------------------------------------------------------------
sub4_pr_aligned = sub4 @ sub4_pr_T
sub5_pr_aligned = sub5 @ sub5_pr_T
sub6_pr_aligned = sub6 @ sub6_pr_T

print(f"Subject 4 Procrustes-aligned shape: {sub4_pr_aligned.shape}")


# ---------------------------------------------------------------------
# Align each subject to the PCA-based common space
# ---------------------------------------------------------------------
sub4_pca_aligned = sub4 @ sub4_pca_T
sub5_pca_aligned = sub5 @ sub5_pca_T
sub6_pca_aligned = sub6 @ sub6_pca_T

print(f"Subject 4 PCA-aligned shape: {sub4_pca_aligned.shape}")

Subject 4 Procrustes-aligned shape: (470, 120)
Subject 4 PCA-aligned shape: (470, 120)


Starting from the next chapter, we introduce the complete ``fMRI-HA`` workflow in detail. For real applications, we recommend using the full workflow described in the following chapters rather than relying only on this minimal quick-start example.

**Demo datasets**

The datasets used in the manual can be downloaded from the links below. Different demo datasets are used for different tutorial sections.

- **Matrix demo dataset**  
  Used primarily in the current demo page for matrix-based examples.  
  [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.20094739.svg)](https://doi.org/10.5281/zenodo.20094739)

- **Movie demo dataset**  
  Includes CIFTI, GIFTI, and NIfTI examples. This dataset is used primarily in the Data Preparation and RHA tutorials.  
  [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.20094820.svg)](https://doi.org/10.5281/zenodo.20094820)

- **Task demo dataset**  
  Used primarily in the CHA and GLM tutorials.  
  [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.20094890.svg)](https://doi.org/10.5281/zenodo.20094890)

- **Task beta-map demo dataset**  
  Used primarily in the MVPA tutorial.  
  [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.20094916.svg)](https://doi.org/10.5281/zenodo.20094916)

- **Supplement data**  
  Includes some supporting data, such as medial-wall masks and related surface resources.  
  [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.20094978.svg)](https://doi.org/10.5281/zenodo.20094978)